## p17 to p27

### p17

In [ ]:
import torch
from PIL import Image
import requests
from transformers import AutoProcessor, CLIPModel
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

In [ ]:
device='cuda'

In [ ]:
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
processor = AutoProcessor.from_pretrained("openai/clip-vit-base-patch32")
model.eval();

In [ ]:
image = Image.open('me_no_hat_cropped_1.jpeg')
text = "A photo of a man"
# image

In [ ]:
inputs = processor(images=image, return_tensors="pt").to(device)
with torch.no_grad():
    image_features_man = model.get_image_features(**inputs)
    # image_features_man = image_features_man / image_features_man.norm(dim=-1, keepdim=True)

In [ ]:
image_features_man[0,:10], image_features_man.shape

In [ ]:
inputs = processor(text=text, return_tensors="pt", padding=True).to(device)
with torch.no_grad():
    text_features_man = model.get_text_features(**inputs)
    # text_features_man = text_features_man / text_features_man.norm(dim=-1, keepdim=True)

In [ ]:
text_features_man[0,:10], text_features_man.shape

In [ ]:
image = Image.open('n02123045_1955.jpg')
text = "A photo of a cat"

inputs = processor(images=image, return_tensors="pt").to(device)
with torch.no_grad():
    image_features_cat = model.get_image_features(**inputs)
    # image_features_cat = image_features_cat / image_features_cat.norm(dim=-1, keepdim=True)

inputs = processor(text=text, return_tensors="pt", padding=True).to(device)
with torch.no_grad():
    text_features_cat = model.get_text_features(**inputs)
    # text_features_cat = text_features_cat / text_features_cat.norm(dim=-1, keepdim=True)

In [ ]:
image = Image.open('n02099601_7101.jpg')
text = "A photo of a dog"

inputs = processor(images=image, return_tensors="pt").to(device)
with torch.no_grad():
    image_features_dog = model.get_image_features(**inputs)
    # image_features_dog = image_features_dog / image_features_dog.norm(dim=-1, keepdim=True)

inputs = processor(text=text, return_tensors="pt", padding=True).to(device)
with torch.no_grad():
    text_features_dog = model.get_text_features(**inputs)
    # text_features_dog = text_features_dog / text_features_dog.norm(dim=-1, keepdim=True)

In [ ]:
cos_similarities=np.zeros((3,3))
dot_products=np.zeros((3,3))
prods=[]

for i, (name_text, feature_text) in enumerate(zip(['cat_text', 'dog_text', 'man_text'], 
                                 [text_features_cat, text_features_dog, text_features_man])):
    prods.append([])
    for j, (name_image, feature_image) in enumerate(zip(['cat_image', 'dog_image', 'man_image'], 
                                     [image_features_cat, image_features_dog, image_features_man])):
        cos_similarities[i,j]=torch.cosine_similarity(feature_image, feature_text)
        
        prod=feature_image*feature_text
        # prod=(feature_image-ave_image_features)*(feature_text-ave_text_features) #Try removing mean to make differences more clear?
        
        dot_products[i,j]=prod.sum().item()
        prods[-1].append(prod.detach().cpu().numpy().reshape(16,32))
        # print(prod.min().item(), prod.max().item())
        

In [ ]:
cos_similarities

In [ ]:
dot_products

In [ ]:
image_features_man.norm(dim=-1)

In [ ]:
text_features_man.norm(dim=-1)